# Spectral Images

A long-slit frame is a photograph of a slit, dragged across the detector by wavelength. That single
sentence explains most of what follows: why the frame has the
structure it does, why the two axes mean completely different things, and why almost every
`specreduce` operation is described as acting "along the dispersion axis" or "across the
cross-dispersion axis".

This chapter is about the 2D frame itself: what its axes are, what the detector puts in them,
and how `specreduce` represents a frame in code. It stops short of the one-dimensional `Spectrum`
container that operations *return*; that is [Chapter 3](03-spectrum-objects.ipynb).

In [ ]:
import sys

sys.path.append("../src")

import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import simple_norm
from specreduce.core import parse_image

from common import parse_section, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data(trim=False)  # this chapter is about the frame as recorded
ny, nx = obj.data.shape

print(f"science frame: {obj.data.shape}  {obj.unit}")

## Two axes, two different meanings

The array is 512 × 1024. Those are not two samplings of the same kind of thing.

- The **dispersion axis** (1024 pixels here) runs along wavelength. Moving along it changes the
  color of the light. A cut along this axis is a spectrum.
- The **cross-dispersion axis** (512 pixels) runs along the slit, i.e. across the sky. Moving
  along it changes *where* on the sky you are looking. A cut along this axis is an image of the
  slit, which for a point source is a seeing disc.

To see the difference, we take one cut of each through the same frame.

In [ ]:
row, column = 275, 512

fig = plt.figure(figsize=(8, 4.6), constrained_layout=True)
grid = fig.add_gridspec(2, 2, height_ratios=[1.15, 1])

frame = fig.add_subplot(grid[0, :])
frame.imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
)
frame.axhline(row, c="C1", lw=0.8)
frame.axvline(column, c="C3", lw=0.8)
frame.set(xlabel="Dispersion [pix]", ylabel="Cross-dispersion [pix]")

along = fig.add_subplot(grid[1, 0])
along.plot(obj.data[row], lw=0.6, c="C1")
along.set(
    xlim=(0, nx - 1),
    xlabel="Dispersion [pix]",
    ylabel="Signal [e$^-$]",
    title=f"row {row}: a spectrum",
)

across = fig.add_subplot(grid[1, 1])
across.plot(obj.data[:, column], lw=0.6, c="C3")
across.set(
    xlim=(0, ny - 1),
    ylim=(10, 2e5),
    yscale="log",
    xlabel="Cross-dispersion [pix]",
    title=f"column {column}: an image of the slit",
)

for ax in (along, across):
    ax.title.set_size(9)

The two panels could hardly look less alike. Along the row you get a broad continuum with
absorption dips, recognizably a stellar spectrum. Along the column you get a flat pedestal of sky
with a narrow spike where the star's light falls, plus some smaller spikes from other objects that
happened to lie in the slit.

Everything the book does follows from this asymmetry. Tracing (Chapter 4) asks *where the spike
sits* as a function of dispersion. Background subtraction (Chapter 7) measures the pedestal either
side of the spike. Extraction (Chapter 8) collapses the spike into one number per column.
Wavelength calibration (Chapters 9–11) puts physical units on the horizontal axis of the left-hand
panel.

## The axis convention

The current release of `specreduce` expects one orientation: **cross-dispersion on axis 0,
dispersion on axis 1**. In NumPy indexing that means `frame[cross_dispersion, dispersion]`: rows
across the slit, columns along wavelength. This is a restriction of the present code rather than a
design principle, and it is due to be relaxed in a future release; for now, a frame in the other
orientation has to be transposed before `specreduce` sees it.

The FITS header, which `CCDData` carries as its `header`, records the orientation as well:

In [ ]:
header = obj.header
print(f"DISPAXIS keyword: {header['DISPAXIS']}")

The `DISPAXIS` header keyword records the same thing in the FITS convention, where axes are
1-based and listed fastest-first, so `DISPAXIS = 1` means the *first* FITS axis (`NAXIS1`, the
1024-pixel one) is the dispersion axis, which is NumPy's **last** axis. The two conventions agree
here; they count from opposite ends.

If your data arrives the other way round, with dispersion running down the columns, transpose it
before handing it to `specreduce`. The frames in this book were transposed when the dataset was
prepared, precisely so that every chapter can take the convention for granted.

:::{note}
Several `specreduce` classes already accept `disp_axis` and `crossdisp_axis` arguments. They are
the interface through which a configurable orientation will arrive, but in the current release the
code behind them still assumes the layout above, so leave them at their defaults and transpose
instead.
:::

## What the detector actually recorded

A frame is not all sky and starlight. Part of it never saw the sky at all, and the FITS header
says which part.

In [ ]:
for keyword in ("NAXIS1", "NAXIS2", "CCDSUM", "DATASEC", "BIASSEC", "TRIMSEC"):
    print(f"  {keyword:8s} = {header[keyword]}")

data_section = parse_section(header["DATASEC"])
bias_section = parse_section(header["BIASSEC"])
print(f"\nDATASEC as numpy slices: rows {data_section[0]}, columns {data_section[1]}")
print(f"BIASSEC as numpy slices: rows {bias_section[0]}, columns {bias_section[1]}")

`BIASSEC` points at the **prescan**: real detector pixels, clocked out and digitized exactly like
the rest, but never exposed to light, which is why the read noise is measured there (Chapter 1).
`DATASEC` marks where the illuminated data begin: row 14 in FITS terms, row 13 in NumPy's 0-based
indexing. `TRIMSEC` is different in kind: it does not describe the hardware but the observatory's
*recommendation* of what to keep, and here it is a conservative one, starting at row 141 (140 in
NumPy) and stopping at column 1000, which drops the last 24 columns at the red end where the
throughput is falling.

None of the three keywords is part of the FITS standard. They are IRAF conventions, written by
observatories because IRAF's tasks read them, and a frame from another instrument may carry all of
them, some, or none. `CCDSUM = '4 4'` records the binning: 2 × 2 on-chip when the data were taken,
and another 2 × 2 applied when this teaching dataset was made, so each pixel here is the sum of
sixteen detector pixels.

`DATASEC` and `TRIMSEC` disagree by 127 rows about where the useful frame starts, and neither is a
measurement. The median along the dispersion axis shows where the light actually is:

In [ ]:
slit_profile = np.median(obj.data, axis=1)
rows = np.arange(ny)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
ax.plot(rows, slit_profile, lw=0.8)

ax.axvspan(bias_section[0].start, bias_section[0].stop, color="C3", alpha=0.18, lw=0)
ax.axvspan(bias_section[0].stop, 70, color="0.6", alpha=0.18, lw=0)
ax.axvspan(70, ny, color="C2", alpha=0.10, lw=0)
ax.axvline(140, c="0.3", ls=":", lw=0.8)

for x, y, label, ha in (
    (3, 3e4, "prescan\n(BIASSEC)", "left"),
    (41, 4e3, "unilluminated", "center"),
    (144, 4e3, "TRIMSEC", "left"),
    (310, 4e3, "illuminated slit", "center"),
):
    ax.text(x, y, label, size=7, ha=ha, c="0.35")

# Symmetric log, so that the zero level of the prescan is on the plot.
ax.set_yscale("symlog", linthresh=10)
ax.set(
    xlim=(0, ny - 1),
    ylim=(-5, 2e5),
    xlabel="Cross-dispersion [pix]",
    ylabel="Median signal [e$^-$]",
);

The prescan sits at zero, as it must once the bias is removed. The strip above it is not
illuminated either, although it lies inside `DATASEC`: it carries a few tens of electrons of
scattered light, rising toward the slit, and the sky does not reach its plateau near 630 e⁻ until
about row 70. `TRIMSEC` does not begin until row 140, some 70 rows after that. Neither keyword
marks the optical edge: `DATASEC` is generous on one side of it and `TRIMSEC` on the other.

Anything that samples the frame along the cross-dispersion axis, such as a background window or a
tilt-correction sample row, has to stay inside the illuminated region. The simplest way to
guarantee that is to cut the frame down once and never think about the edge again, and that is
what we do. From the next chapter on, `read_data` trims every frame to `TRIMSEC`. Nothing of value
goes with it, only 70 rows of sky on a 40-arcsecond slit and two dozen columns beyond the grism's
red cutoff, so every later chapter works on a 372 × 1000 frame with the target near row 135
instead of row 275.

## How `specreduce` reads a frame

Every operation in the package funnels its input through one function,
`specreduce.core.parse_image`. It accepts a `Spectrum`, a `CCDData`, an `NDData`, a `Quantity` or a
plain `ndarray`, and returns a `Spectrum` with a unit, an uncertainty, a spectral axis and a mask
all present, so that the classes downstream never have to check what they were given. The
`CCDData` that `read_data` returns carries a unit (electrons, from `BUNIT`) and a variance, and
`parse_image` passes both through unchanged; what it does with an input that lacks them is the
subject of the next chapter. Every chapter here loads through `read_data`, so its frames arrive
fully specified.

## Masks

The last thing `parse_image` normalizes is the mask, and it takes a `mask_treatment` argument
controlling how masked and non-finite pixels are handled. Real frames need this: dead pixels,
saturated cores and cosmic rays all have to be excluded from fits without being silently replaced
by nonsense.

We exercise the whole option set on a copy of the science frame with a single bad pixel and part
of a bad column blanked out.

In [ ]:
damaged = np.array(obj.data, dtype=float)
damaged[300, 400] = np.nan  # one dead pixel
damaged[100:110, 700] = np.nan  # part of a bad column

print(f"non-finite pixels introduced: {int(np.isnan(damaged).sum())}\n")
print(
    f"{'mask_treatment':>16}  {'masked':>7}  {'non-finite left':>16}  {'value at [300, 400]':>20}"
)
for treatment in (
    "apply",
    "ignore",
    "propagate",
    "zero_fill",
    "nan_fill",
    "apply_mask_only",
    "apply_nan_only",
):
    parsed = parse_image(damaged, mask_treatment=treatment)
    mask = (
        np.asarray(parsed.mask)
        if parsed.mask is not None
        else np.zeros(damaged.shape, bool)
    )
    values = np.asarray(parsed.data)
    print(
        f"{treatment:>16}  {int(mask.sum()):>7}  {int(np.isnan(values).sum()):>16}  "
        f"{values[300, 400]:>20.1f}"
    )

Reading that table:

- **`apply`** (the default) masks the non-finite pixels and leaves the data alone. This is what you
  want almost always.
- **`ignore`** drops the mask entirely; the NaNs are still in the data, now unflagged. Use it only
  when you know the frame is clean.
- **`propagate`** masks the *entire cross-dispersion column* containing any bad pixel: 11 bad
  pixels become 1024 masked ones, two full columns. Drastic, but appropriate when a fit would be
  biased by a partially sampled column.
- **`zero_fill`** and **`nan_fill`** substitute a fill value and drop the mask. `zero_fill` is
  dangerous in exactly the way it sounds: a zero is a perfectly valid flux value, so the bad pixel
  becomes invisible and biases every sum it enters. `nan_fill` is indistinguishable from `ignore`
  in this demo, because the bad pixels were already NaN, but the two differ when the input carries a mask
  over *finite* pixels: `nan_fill` blanks those to NaN, `ignore` drops them.
- **`apply_mask_only`** honors an existing mask but ignores non-finite values. Note it leaves all
  11 NaNs unmasked here, because the input array carried no mask of its own.
- **`apply_nan_only`** does the reverse: builds a fresh mask from the non-finite values and discards
  any mask that came in.

Not every class supports every option. `FitTrace`, for instance, accepts only `apply`,
`propagate` and `apply_nan_only`, because its fit cannot consume non-finite values at all.

## Summary

- A long-slit frame has two axes with different physical meanings: **dispersion** (wavelength, a
  cut along it is a spectrum) and **cross-dispersion** (position along the slit, a cut along it is
  an image of the slit). Every operation in the book is described in terms of one axis or the
  other.
- The current release of `specreduce` expects **cross-dispersion on axis 0, dispersion on
  axis 1**, that is `frame[cross_dispersion, dispersion]`. Transpose your data if it arrives
  otherwise. The `disp_axis` and `crossdisp_axis` arguments exist but are not yet functional, so
  leave them at their defaults.
- The FITS `DISPAXIS` keyword records the same orientation in 1-based, fastest-first FITS
  ordering, so `DISPAXIS = 1` is NumPy's last axis.
- `BIASSEC`, `DATASEC` and `TRIMSEC` mark the prescan, the data region and the observatory's
  recommended trim. The prescan never sees light, which is what makes it the place to
  measure the read noise.
- From Chapter 3 on, `read_data` **trims every frame to `TRIMSEC`**, so the later chapters work
  on a 372 × 1000 frame with the target near row 135 and never have to steer around the edge. The
  trim is detector-level bookkeeping and lives in `src/common.py`.
- `parse_image` normalizes every input (`Spectrum`, `CCDData`, `NDData`, `Quantity`, `ndarray`)
  into one `Spectrum` with a unit, an uncertainty, a spectral axis and a mask, which is what every
  operation in the package consumes.
- `mask_treatment` controls how masked and non-finite pixels are handled. `apply` is the sensible
  default, `propagate` masks whole columns, `zero_fill` hides bad pixels inside valid-looking
  values, and not every class accepts every option.

Next: [Spectrum Objects](03-spectrum-objects.ipynb), the container that `specreduce` operations
consume and return.
